# N148 · 随机集合与多重集合设计

**目标：** 用交换末尾删除维持稠密数组及反向索引。

**先修：** N017, N019, N131。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 数组+哈希；平均O(1)；重复值索引集合；均匀随机索引。

## 从问题到算法

随机取样要求能O(1)访问随机下标，快速删除要求能O(1)定位元素。用数组加哈希映射，把要删除的槽位用末项填补，再更新末项的位置。允许重复值时，value对应的不是单个下标，而是一组下标；特别要检查被移来的末项与被删值相同的情况。

## 最小实现

**本章接口：** `RandomizedSet`、`RandomizedCollection`

In [1]:
from random import Random

class RandomizedSet:
    def __init__(self,rng=None): self.values=[]; self.position={}; self.rng=Random() if rng is None else rng
    def insert(self,value):
        if value in self.position: return False
        self.position[value]=len(self.values); self.values.append(value); return True
    def remove(self,value):
        if value not in self.position: return False
        index=self.position.pop(value); last=self.values.pop()
        if index<len(self.values): self.values[index]=last; self.position[last]=index
        return True
    def getRandom(self):
        if not self.values: raise IndexError('empty randomized set')
        return self.values[self.rng.randrange(len(self.values))]

class RandomizedCollection:
    def __init__(self,rng=None): self.values=[]; self.positions={}; self.rng=Random() if rng is None else rng
    def insert(self,value):
        first=value not in self.positions
        self.positions.setdefault(value,set()).add(len(self.values)); self.values.append(value); return first
    def remove(self,value):
        if value not in self.positions: return False
        index=self.positions[value].pop(); last=self.values[-1]; last_index=len(self.values)-1
        self.values[index]=last; self.positions[last].add(index); self.positions[last].discard(last_index); self.values.pop()
        if not self.positions[value]: del self.positions[value]
        return True
    def getRandom(self):
        if not self.values: raise IndexError('empty randomized collection')
        return self.values[self.rng.randrange(len(self.values))]

## 正确性、前提与复杂度

交换删除只改变一个保留元素的位置，因此仅需常数个映射更新；数组始终紧凑，均匀随机下标等价于均匀随机元素实例。多重集合中某值被选中概率正比于出现次数，而不是对不同值均匀。索引集合增删顺序处理末项同值和删除末槽两种边界。

**代价：** 插入、删除与随机取样平均/均摊O(1)，空间O(元素实例数)。getRandom在空结构上抛IndexError；不通过返回任意默认值掩盖非法调用。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

collection=RandomizedCollection(Random(148)); rows=[]
for op,value in [('insert',1),('insert',1),('insert',2),('remove',1),('remove',2)]:
    result=getattr(collection,op)(value); rows.append((op,value,result,collection.values[:],{k:sorted(v) for k,v in collection.positions.items()}))
show_table(['操作','值','返回','数组','索引集合'],rows)

操作,值,返回,数组,索引集合
insert,1,True,[1],{1: [0]}
insert,1,False,"[1, 1]","{1: [0, 1]}"
insert,2,True,"[1, 1, 2]","{1: [0, 1], 2: [2]}"
remove,1,True,"[2, 1]","{1: [1], 2: [0]}"
remove,2,True,[1],{1: [0]}


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from collections import Counter
rng=Random(148); a=RandomizedSet(Random(1)); b=RandomizedCollection(Random(1)); ref=set(); multiset=Counter()
for _ in range(1200):
    x=rng.randrange(10)
    if rng.randrange(2):
        assert a.insert(x)==(x not in ref); ref.add(x)
        assert b.insert(x)==(multiset[x]==0); multiset[x]+=1
    else:
        assert a.remove(x)==(x in ref); ref.discard(x)
        assert b.remove(x)==(multiset[x]>0)
        if multiset[x]: multiset[x]-=1
        if multiset[x]==0: multiset.pop(x,None)
    assert set(a.values)==ref and len(a.values)==len(ref)
    assert a.position=={x:i for i,x in enumerate(a.values)}
    assert Counter(b.values)==multiset
    assert b.positions=={x:{i for i,y in enumerate(b.values) if y==x} for x in multiset}
    if ref: assert a.getRandom() in ref
    if multiset: assert b.getRandom() in multiset
class Tickets:
    def __init__(self): self.next=0
    def randrange(self,n): result=self.next; self.next+=1; return result
b=RandomizedCollection(Tickets())
for x in [1,1,2]: b.insert(x)
assert Counter(b.getRandom() for _ in range(3))==Counter({1:2,2:1})
print('N148: 所有本章断言通过')

N148: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释按不同值均匀与按出现次数均匀的差别。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 随机数源作为参数注入。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 380. Insert Delete GetRandom O(1)（canonical）
- 381. Insert Delete GetRandom O(1) - Duplicates allowed（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。